# Scaling Curves & Emergence Analysis

This notebook walks through generating publication-quality scaling curve plots
from eval results. It demonstrates:

1. Loading results into a DataFrame
2. Plotting individual scaling curves (accuracy and logprob)
3. Two-panel emergence comparison (Schaeffer-style)
4. Fitting emergence thresholds with bootstrap confidence intervals

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sse.analysis import (
    PYTHIA_PARAMS,
    fit_emergence_threshold,
    plot_emergence_comparison,
    plot_scaling_curve,
)
from sse.results import load_all_results

%matplotlib inline

## Load Results

Load all JSONL results from the results directory. If no real results exist yet,
we generate synthetic data that mimics expected emergence patterns.

In [ ]:
results_dir = Path("../results")

if results_dir.exists() and list(results_dir.glob("*.jsonl")):
    df = load_all_results(results_dir)
    print(f"Loaded {len(df)} rows from {results_dir}")
else:
    # Generate synthetic data for demonstration
    sizes = ["70m", "160m", "410m", "1b", "1.4b", "2.8b"]
    params = [PYTHIA_PARAMS[s] for s in sizes]
    log_params = np.log10(params)

    # Sigmoid accuracy: near 0 for small models, rising to ~0.85 for large
    accuracy = 0.85 / (1 + np.exp(-3.0 * (log_params - 8.8)))
    # Linear logprob: improves (less negative) with scale
    logprob = -12 + 1.5 * (log_params - 7.8)

    rows = []
    for i, size in enumerate(sizes):
        base = {
            "run_id": f"synthetic-{i}",
            "timestamp": "2025-01-01T00:00:00Z",
            "model_size": size,
            "model_revision": "main",
            "task_name": "arithmetic",
            "task_version": "v1",
            "n_examples": 200,
            "schema_version": "1.0",
        }
        rows.append({
            **base, "metric_name": "accuracy",
            "metric_value": float(accuracy[i]),
        })
        rows.append({
            **base, "metric_name": "mean_logprob_correct",
            "metric_value": float(logprob[i]),
        })

    df = pd.DataFrame(rows)
    print(f"Using synthetic data ({len(df)} rows, {len(sizes)} model sizes)")

df.head(10)

## Available Tasks and Models

In [ ]:
print("Tasks:", df["task_name"].unique().tolist())
print("Metrics:", df["metric_name"].unique().tolist())
print("Model sizes:", df["model_size"].unique().tolist())

## Scaling Curve: Accuracy

Plot accuracy vs model parameters on a log scale. The sigmoid fit captures
the discontinuous emergence pattern — near-chance for small models, then
a sharp transition to high accuracy.

In [ ]:
task = df["task_name"].iloc[0]

ax = plot_scaling_curve(df, task, "accuracy")
plt.show()

## Scaling Curve: Log-probability

The logprob metric improves linearly in log-parameter space — no sudden
transition. This is the key insight from Schaeffer et al.: emergence is an
artifact of the evaluation metric, not a fundamental property of the model.

In [ ]:
ax = plot_scaling_curve(df, task, "mean_logprob_correct")
plt.show()

## Emergence Comparison (Two-Panel)

Side-by-side comparison showing the same task measured with both metrics.
Left panel shows the discontinuous (accuracy) view; right panel shows the
continuous (logprob) view.

In [ ]:
fig = plot_emergence_comparison(df, task)
plt.show()

## Emergence Threshold with Bootstrap CI

Fit a sigmoid to the accuracy curve and find the model size at which
performance crosses a given threshold (e.g., 40% accuracy). Bootstrap
resampling gives a 95% confidence interval on this estimate.

In [ ]:
result = fit_emergence_threshold(df, task, threshold=0.4, n_bootstrap=500)

print(f"Task: {task}")
print("Threshold: 40% accuracy")
print(f"Estimated crossing: {result['threshold_params']:.2e} parameters")
print(f"95% CI: [{result['ci_lower']:.2e}, {result['ci_upper']:.2e}]")

## Save Figures

Save all main figures to `results/figures/` for use in reports.

In [ ]:
figures_dir = Path("../results/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

plot_scaling_curve(df, task, "accuracy", output_dir=figures_dir)
plt.close()

plot_scaling_curve(df, task, "mean_logprob_correct", output_dir=figures_dir)
plt.close()

plot_emergence_comparison(df, task, output_dir=figures_dir)
plt.close()

print(f"Figures saved to {figures_dir.resolve()}:")
for f in sorted(figures_dir.glob("*")):
    print(f"  {f.name}")